In [0]:
import data.utilities.common as Commonlib
import yaml
from beertech_utils.data.integrity.integrity_check import EmptyDataFrameCheck
from data.utilities.extractors import OracleExtractor
from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from pyspark.sql.functions import col

In [0]:
# this entire command block will re-execute each time the "1. Subject Area" drop-down selection is changed - this populates the appropriate configurations in the "2. Configuration File" drop-down
CONFIG_BASE_PATH = "../../configuration/oracle/"

subject_entries = Commonlib.list_directories(CONFIG_BASE_PATH)
dbutils.widgets.dropdown("subject_area", "", [""] + sorted(subject_entries), "1. Subject Area")
subject_area = dbutils.widgets.get("subject_area")

config_selection_path = f"{CONFIG_BASE_PATH}{subject_area}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "2. Configuration File")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}{subject_area}/{config_file}"

config = Commonlib.get_object_config(config_file_path)
source_config = config.get("source")
bronze_config = config.get(ML.bronze)

In [0]:
try:
    medallion = Medallion(config)

    oracle_schema = source_config.get("schema")
    oracle_table = source_config.get("name", config.get("name", ""))
    oracle_query = source_config.get("query", "")
    oracle_fqtn = f"{oracle_schema}.{oracle_table}"
    oracle_secret_scope = source_config.get("secret_scope", "alchemy-kv")

    oracle_extract = OracleExtractor(secret_key=source_config.get("connection"), secret_scope=oracle_secret_scope)

    if oracle_query:
        medallion.df = oracle_extract.query(oracle_query)
    else:
        medallion.df = oracle_extract.load(oracle_fqtn)

    # destroy oracle extractor object
    del oracle_extract
    # cache the result of the first evaluation
    medallion.df.cache()

    watermark_col = bronze_config.get("watermark", "")
    load_type = bronze_config.get("load_type")

    # incremental load - based on watermark
    if load_type == "append" and watermark_col:
        watermark = medallion.get_watermark(layer=ML.bronze, watermark_col=watermark_col)
        medallion.df = medallion.df.filter(col(watermark_col) > watermark)
        medallion.logger.info(
            f"Performing incremental data load from oracle based on records where {watermark_col} > {watermark}"
        )
    # full load
    elif load_type == "overwrite":
        empty_df = EmptyDataFrameCheck(dataset_name=medallion.name)
        empty_df.run(medallion.df)
        medallion.logger.info("Performing full data load from oracle")
    # custom query load
    else:
        medallion.logger.info("Appending records from oracle based on custom query")
except Exception as e:
    medallion.logger.error(
        f"An error occured while processing the oracle_procedure job. subject={subject_area}, config_file={config_file}: {e}"
    )
    raise

In [0]:
try:
    medallion.write_to_delta(
        load_type=load_type,
        layer=ML.bronze,
        source=f"OracleExtract | {oracle_fqtn}",
    )

    medallion.df.unpersist()
except Exception as e:
    medallion.logger.error(
        f"An error occured while writing to bronze via oracle_procedure. subject={subject_area}, config_file={config_file}: {e}"
    )
    raise